In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../")
import data_loading as dl
import cut_flow as cf
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections
from microfit.xsec_signal_generator import XsecCovarHistGenerator

In [3]:
keep_vars = [
    "Signal_1e1p", "mc_signal_1e1p", "nu_pdg", "TrueElecIdx", "TrueLeadProtonIdx", "InFV", "HasNoMesons", "TrueNElec", "TrueNProt", 
    "TrueDeltaPT", "TrueDeltaAlphaT", "TruePN", "TrueAlpha3D",
    "TrueLeadProtonKE", "TrueLeadProtonModMom", "TrueLeadProtonE", "TrueLeadProtonMomX", "TrueLeadProtonMomY", "TrueLeadProtonMomZ",
    "TrueElecKE", "TrueElecModMom", "TrueElecE", "TrueElecMomX", "TrueElecMomY", "TrueElecMomZ",

    "Sel_1e1p", "sel_1e1p_w_cuts", "RecoElectronCandidateIdx", "RecoLeadProtonCandidateIdx", "InFV_reco",
    "RecoElecPassMomCut", "RecoLeadProtonPassMomCut", "n_reco_tracks", "n_reco_showers",
    "RecoDeltaPT", "RecoDeltaAlphaT", "RecoPN", "RecoAlpha3D", "RecoECal", "Reco_mag_q", "RecoPL",
    "RecoLeadProtonKE", "RecoLeadProtonModMom", "RecoLeadProtonMomX", "RecoLeadProtonMomY", "RecoLeadProtonMomZ",
    "RecoElecE", "RecoElecModMom", "RecoElecMomX", "RecoElecMomY", "RecoElecMomZ",

    "RecoLeadProton_trk_len", "RecoLeadProton_trk_trunk_dEdx_y", "RecoLeadProton_dEdx_y_per_trklen",
    "RecoLeadProtonCandidate_trk_pid", "RecoElectronCandidate_shr_pid", "RecoElectron_conversion_dist",

    "nproton", "npion", "npi0", "nelec", "nmuon", "isVtxInFiducial",

    "nslice", "selected", "shr_energy_tot_cali", "_opfilter_pe_beam", "_opfilter_pe_veto", "bnbdata", "extdata",
    "CosmicIPAll3D", "hits_ratio", "shrmoliereavg", "subcluster", "trkfit", "trkshrhitdist2", "tksh_distance",
    "shr_tkfit_nhits_tot", "shr_tkfit_dedx_max", "tksh_angle", "shr_trk_len", "reco_e",
    "trkpid", "trk_len", "n_showers_contained", "protonenergy_corr", "n_tracks_contained",
    "pi0_radlen1", "pi0_radlen2", "pi0_score", "nonpi0_score", "bkg_score",

    "InFV_1muNp", "TrueMuonIdx_1muNp", "TrueLeadProtonIdx_1muNp", "TrueNProt_1muNp", "TrueFSPions_1muNp", "Signal_1mu1p", 
    "TrueDeltaPT_1mu1p", "TrueDeltaAlphaT_1mu1p", "TruePN_1mu1p", "TrueAlpha3D_1mu1p",
    "TrueLeadProtonE_1muNp", "TrueLeadProtonMomX_1muNp", "TrueLeadProtonMomY_1muNp", "TrueLeadProtonMomZ_1muNp",
    "TrueMuonE_1muNp", "TrueMuonMomX_1muNp", "TrueMuonMomY_1muNp", "TrueMuonMomZ_1muNp",

    "sel_CC1p0pi", "InFV_reco_1muNp", "MuonCandidateIdx_1muNp", "LeadProtonIdx_1muNp", "LeadProtonPassMomentumCut_1muNp", "PFPStartsInPCV_1muNp", "PassTopoScoreCut_1muNp",
    "PassNuMuCCSelection_1muNp", "NoRecoShowers_1muNp", "MuonContained_1muNp", "PassMuonMomentumCut_1muNp", "PassMuonQualCut_1muNp",
    "LeadProtonPassMomentumCut_1muNp", "NProtons_1muNp",
    "RecoDeltaPT_1mu1p", "RecoDeltaAlphaT_1mu1p", "RecoPN_1mu1p", "RecoAlpha3D_1mu1p", "RecoECal_1mu1p", "RecoPL_1mu1p",
    "RecoLeadProtonE_1muNp", "RecoLeadProtonMomentum_1muNp", "RecoLeadProtonMomX_1muNp", "RecoLeadProtonMomY_1muNp", "RecoLeadProtonMomZ_1muNp", 
    "RecoMuonE_1muNp", "RecoMuonMomentum_1muNp", "RecoMuonMomX_1muNp", "RecoMuonMomY_1muNp", "RecoMuonMomZ_1muNp",
]

In [4]:
RUN = ["3"]
#RUN = ["1","2","3_nocrt","3_crt","4a","4b","4c","4d","5"]
#RUN = ["1","2","3","4c","5"] # for nuwro_fd, no run 4b and 4d available
#RUN = ["1","2","3","4a","4b","4c","4d","5","1A_OT","1B_OT"]
#RUN = ["3","4a","4b","4c","4d","5"]
blinded = True

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=True,
    load_lee=False,
    #load_numu_tki=True,
    load_nue_tki=True,
    keep_columns=keep_vars,
    blinded=True,
    load_crt_vars=False,
    enable_cache=True,
)

Loading run 3
Updating keep_columns with truth-filtering variables: {'nu_pdg', 'ccnc'}


/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/bnb_beam_off_peleeTuple_uboone_v08_00_00_70_run3.root
Loading data from: /exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/bnb_beam_off_peleeTuple_uboone_v08_00_00_70_run3.root
Calc true TKI variables for leading proton only
Calc true GKI variables for leading proton only
Calc reco TKI variables for leading proton only
Calc reco GKI variables for leading proton only


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->boolean,key->block6_values] [items->Index(['Signal_1e1p', 'Sel_1e1p'], dtype='object')]

  encoding=encoding,


/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root
Loading data from: /exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root


../../data_loading.py:876: RuntimeWarning: invalid value encountered in true_divide
  df["proton_pz"] = np.where((mc_E_prot > 0), mc_pz_prot / mc_p_prot, np.nan)


Calc true TKI variables for leading proton only
Calc true GKI variables for leading proton only
Calc reco TKI variables for leading proton only
Calc reco GKI variables for leading proton only


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block3_values] [items->Index(['weightsReint', 'weightsGenie', 'weightsFlux', 'Signal_1e1p',
       'Sel_1e1p'],
      dtype='object')]

  encoding=encoding,


/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nue.root
Loading data from: /exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nue.root
Calc true TKI variables for leading proton only
Calc true GKI variables for leading proton only
Calc reco TKI variables for leading proton only
Calc reco GKI variables for leading proton only
/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_dirt.root
Loading data from: /exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_dirt.root
Calc true TKI variables for leading proton only
Calc true GKI variables for leading proton only
Calc reco TKI variables for leading proton only
Calc reco GKI variables for leading proton only


In [5]:
run_combo = "Run"
for run in RUN:
    run_combo += run

In [23]:
preselection = "NUE"
selection = "LucileSEL"
cut_dictionary = cf.do_cut_flow(rundata, preselection, selection,'nue', printed=True)
# cut_dictionary[key][cut][excluded] is a dataframe containing events excluded by the cut,
# similarly cut_dictionary[key][cut][included] is a dataframe containing events included (order matters here!)

Step      Cut Condition                 nue       Efficiency
------------------------------------------------------------
Cut.0     No Cuts                       237.18    100.00%   
------------------------------------------------------------
Pre.1     nslice == 1                   237.18    100.00% 
Pre.2     selected == 1                 237.18    100.00% 
Pre.3     shr_energy_tot_cali > 0.07    237.18    100.00% 
Pre.4     ( (_opfilter_pe_beam > 0 and  234.03     98.67% 
          _opfilter_pe_veto < 20) or    
          bnbdata == 1 or extdata == 1) 
------------------------------------------------------------
Sel.1     CosmicIPAll3D > 10.           230.87     97.34% 
Sel.2     trkpid<(0.015*trk_len+0.02)   135.08     56.95% 
Sel.3     hits_ratio > 0.50             122.98     51.85% 
Sel.4     shrmoliereavg < 9             95.82      40.40% 
Sel.5     subcluster > 4                94.09      39.67% 
Sel.6     trkfit < 0.65                 84.00      35.42% 
Sel.7     shr_trk_len <

In [24]:
print(rundata)

{'data': None, 'ext':        nproton  CosmicIPAll3D  selected  npion  nelec       trk_len  \
entry                                                                 
0            0      82.917274         1      0      0 -3.402823e+38   
1            0      31.453773         1      0      0 -3.402823e+38   
2            0      30.533966         1      0      0 -3.402823e+38   
3            0      93.816269         1      0      0 -3.402823e+38   
4            0       1.936272         1      0      0 -3.402823e+38   
...        ...            ...       ...    ...    ...           ...   
10348        0      16.108452         1      0      0  9.776535e+01   
10349        0      19.095554         1      0      0 -3.402823e+38   
10350        0      20.137905         1      0      0  1.366366e+00   
10351        0      31.524496         1      0      0  7.978885e-01   
10352        0      47.203094         1      0      0  1.730292e+00   

       shr_energy_tot_cali  _opfilter_pe_beam  trkshrh

In [7]:
print(cut_dictionary)

{'nue': {'nslice == 1': {'included':        knobCCMECup  knobAxFFCCQEdn  knobDecayAngMECdn  knobRPAdn  \
entry                                                              
0         1.165609        1.165609           1.165609   1.204096   
1         1.193517        1.193517           1.193517   1.084858   
2         1.273863        1.273863           1.273863   1.439152   
3         1.172451        1.172451           1.172451   0.954694   
4         1.206837        1.206837           1.206837   0.808219   
...            ...             ...                ...        ...   
61952     1.173020        1.173020           1.173020   1.096674   
61953     1.141592        1.141592           1.141592   0.840117   
61954     0.964214        0.964214           0.964214   1.093812   
61955     1.181001        1.181001           1.181001   1.240613   
61956     1.000000        1.000000           1.000000   1.000000   

       knobVecFFCCQEup  nproton  CosmicIPAll3D  knobThetaDelta2Npiup  \
entry 